In [1]:
# ============================================================================
# Negation Awareness CLIP - Main Notebook (Modular Version)
# ============================================================================
# This notebook now uses modular imports from src/ instead of inline code
# All classes and functions are imported from their respective modules
# ============================================================================

# Setup
import torch
import json
import os
from torch.utils.data import DataLoader
import clip
from tqdm import tqdm

# Import all modules
from src.data import (
    NegationJSONDataset, 
    COCOValLlamaDataset, 
    NegRefCOCOgDataset, 
    VALSEDataset
)
from src.features import FeatureCache, extract_and_cache
from src.llm import LocalQwenClient, SubQueryExtractor
from src.models import DEOModel, NegationSteeredCLIP, load_negation_direction
from src.training import train_binary_negation_classifier, steer_embeddings
from src.evaluation import PairwiseModelAdapter, evaluate_pairwise_preference
from src.experiments import run_paper_negation_experiment, evaluate_negation_steering_on_text

print("✓ All modules imported successfully!")

/home/kritic/miniconda3/envs/dlcv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Negation Awareness CLIP - Modular Version

This notebook has been **refactored to use modular imports** instead of inline code.

## What Changed?

✅ **Removed 600+ lines** of inline class definitions  
✅ **Now imports from modular structure** (`src/data/`, `src/models/`, etc.)  
✅ **Cleaner, more maintainable code**  
✅ **Easier to test and extend**  

## Module Structure

- **`src/data/`** - Dataset classes (NegationJSON, COCOValLlama, NegRefCOCOg, VALSE)
- **`src/features/`** - FeatureCache and embedding extraction
- **`src/llm/`** - LocalQwenClient and SubQueryExtractor
- **`src/models/`** - DEOModel and NegationSteeredCLIP
- **`src/training/`** - train_binary_negation_classifier and steer_embeddings
- **`src/evaluation/`** - Model adapters and metrics
- **`src/experiments/`** - Main experiment pipelines

All imports are at the top of this notebook. No more copy-pasted code!


In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Using device: {device}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Download COCO images first:
# wget http://images.cocodataset.org/zips/train2014.zip
# unzip train2014.zip

# Then load dataset:
# dataset = NegRefCOCOgDataset(
#     json_path="NegRefCOCOg.json",
#     images_dir="./train2014"
# )

# loader = DataLoader(dataset, batch_size=8)
# batch = next(iter(loader))
# batch['text'] - list of phrases
# batch['crop_ref'] - tensor [B, 3, 224, 224]
# batch['crop_alt'] - tensor [B, 3, 224, 224]

In [6]:
dataset = COCOValLlamaDataset("COCO_val_mcq_llama3.1_rephrased.json", max_samples=14000)
loader = DataLoader(dataset, batch_size=128, shuffle=False)

In [ ]:
dataset = NegationJSONDataset("negationclip_captions_train2014.json", max_samples=14000)
loader = DataLoader(dataset, batch_size=128, shuffle=False)

In [57]:
dataset = NegRefCOCOgDataset("NegRefCOCOg.json", max_samples=7000)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [121]:
dataset = NegationJSONDataset("gemini_negation_dataset.json", max_samples=14000)
loader = DataLoader(dataset, batch_size=128, shuffle=False)

In [6]:
print(f"Dataset length: {len(dataset)}")
dataset[0]

Dataset length: 5914


{'pos_text': 'No bowl is present in this image.',
 'neg_text': 'This image contains a bowl, but no knife is present.',
 'image_id': 'data/coco/images/val2017/000000209530.jpg'}

In [7]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": -1}
base_model, _ = clip.load("ViT-B/32", device=device)
base_features = extract_and_cache(base_model, loader, clip.tokenize, "Baseline_CLIP_COCOValLlama", base_config, dataset="COCOValLlama", device=device)

--- Found cached features: ./embeddings_cache/Baseline_CLIP_COCOValLlama_dde73a68053404cba0e24193f3a00875.pt ---


/tmp/ipykernel_368465/3879348306.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path)


In [111]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": -1}
base_model, _ = clip.load("ViT-B/32", device=device)
base_features = extract_and_cache(base_model, loader, clip.tokenize, "Baseline_CLIP_14000", base_config, device=device)

--- Found cached features: ./embeddings_cache/Baseline_CLIP_14000_d348be7ef9e86f5952b8187a03a6cdac.pt ---


/tmp/ipykernel_273155/3879348306.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path)


In [128]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": 4}
base_model, _ = clip.load("ViT-B/32", device=device)
base_features = extract_and_cache(base_model, loader, clip.tokenize, "Baseline_CLIP_Gemini", base_config, dataset="GEMINI_NEGATION", device=device)

Encoding Baseline_CLIP_Gemini (Layer 4): 100%|██████████| 79/79 [00:03<00:00, 26.18it/s]

--- Features successfully cached at ./embeddings_cache/Baseline_CLIP_Gemini_69e9c1072331f9acad27b90f6f251cc6.pt ---


In [21]:
layers_to_save = range(1, 13) 
for l in layers_to_save:
    current_cfg = {
        "type": "vanilla", 
        "arch": "ViT-B/32", 
        "layer": l
    }
    
    variant_name = "Baseline_CLIP_COCOValLlama"
    
    print(f"\n>>> Extracting features for Layer {l}...")
    
    extract_and_cache(
        model=base_model, 
        dataloader=loader, 
        dataset = "COCOValLlama",
        tokenizer=clip.tokenize, 
        model_variant_name=variant_name, 
        config=current_cfg,
        device="cuda"
    )

print("\n All 12 layers have been cached successfully.")


>>> Extracting features for Layer 1...


Encoding Baseline_CLIP_COCOValLlama (Layer 1): 100%|██████████| 47/47 [00:01<00:00, 39.07it/s]


--- Features successfully cached at ./embeddings_cache/Baseline_CLIP_COCOValLlama_cdb9f7c5ffea5d1441f91f047b78df6f.pt ---

>>> Extracting features for Layer 2...


Encoding Baseline_CLIP_COCOValLlama (Layer 2): 100%|██████████| 47/47 [00:01<00:00, 37.01it/s]


--- Features successfully cached at ./embeddings_cache/Baseline_CLIP_COCOValLlama_a4354d7505da586a7faa3b8045d2a167.pt ---

>>> Extracting features for Layer 3...


...

--- Features successfully cached at ./embeddings_cache/Baseline_CLIP_COCOValLlama_076eac8f1d367d9a9535bfba98c7fec0.pt ---

>>> Extracting features for Layer 11...


Encoding Baseline_CLIP_COCOValLlama (Layer 11): 100%|██████████| 47/47 [00:02<00:00, 16.34it/s]


--- Features successfully cached at ./embeddings_cache/Baseline_CLIP_COCOValLlama_610a226dfc1e34f8ee839dd508e73de4.pt ---

>>> Extracting features for Layer 12...


Encoding Baseline_CLIP_COCOValLlama (Layer 12): 100%|██████████| 47/47 [00:03<00:00, 15.50it/s]

--- Features successfully cached at ./embeddings_cache/Baseline_CLIP_COCOValLlama_608ac0db4759af14d53b8e8c854ec37c.pt ---

 All 12 layers have been cached successfully.


In [14]:
# Update your extractor to use this new client
extractor = SubQueryExtractor(
    llm_client=qwen_client, 
    model_name=qwen_client.model_id, 
    prompt_version="v3",
    dataset_name="COCOValLlama"
)

In [ ]:
# LLM Decomposition - Extract positives and negatives for all queries
print(f"Starting LLM decomposition for {len(loader.dataset)} samples...")

# Get unique queries from dataset
all_queries = []
for batch in tqdm(loader, desc="Scanning Dataset"):
    if "pos_text" in batch and "neg_text" in batch:
        all_queries.extend(batch['pos_text'])
        all_queries.extend(batch['neg_text'])

unique_queries = list(set(all_queries))
print(f"Total Unique Queries: {len(unique_queries)}")

# Process through SubQueryExtractor (uses LLM with caching)
batch_size = 8
for i in tqdm(range(0, len(unique_queries), batch_size), desc="GPU Batch Inference"):
    chunk = unique_queries[i : i + batch_size]
    try:
        extractor.get_decomposition_batch(chunk)
    except Exception as e:
        print(f"Error at index {i}: {e}")

print(f"✓ LLM decomposition complete. Cache at: {extractor.cache_dir}")

In [20]:
extract_all_decompositions(loader, batch_size=8)

Starting batched decomposition for 47 samples (Pos + Neg paths)...


Scanning Dataset: 100%|██████████| 47/47 [00:00<00:00, 4108.72it/s]


Total Unique: 7651 | Cached: 0 | To Process: 7651


GPU Batch Inference:   1%|          | 9/957 [00:31<1:28:46,  5.62s/it]

Extraction Error for: "A car is absent from this ima... trying simple fallback.


...

Extraction Error for: This image features a pizza, b... trying simple fallback.


GPU Batch Inference:  98%|█████████▊| 942/957 [46:12<01:28,  5.92s/it]

Extraction Error for: A chair is present in this ima... trying simple fallback.


GPU Batch Inference: 100%|██████████| 957/957 [46:46<00:00,  2.93s/it]


--- Extraction Summary ---
Total Unique Queries: 7651
Loaded from Cache:    0
Newly Extracted:      7651
Failed:               0
Cache location:       ./llm_cache/COCOValLlama/v3


In [16]:
deo_baseline_model = DEOModel(base_model, extractor, config={
    "lr": 0.01,
    "steps": 20,
    "pos_weight": 1.0,
    "neg_weight": 1.0,
    "reg_weight": 0.2
})
deo_base_config = {"type": "vanilla", "arch": "ViT-B/32"}
deo_base_features = extract_and_cache(deo_baseline_model, loader, clip.tokenize, "DEO_CLIP", deo_base_config, device=device)

Encoding DEO_CLIP (Layer -1): 100%|██████████| 47/47 [03:59<00:00,  5.09s/it]

--- Features successfully cached at ./embeddings_cache/DEO_CLIP_6673ee85278670e3ea4346a14ee96680.pt ---


In [ ]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": 4}

# If you want to compare against your DEO features, use that config
# Otherwise, you can compare Baseline Pos vs Baseline Neg
exp_config = {
    'dataset': "NegationCLIP",
    'pos_variant': 'Baseline_CLIP',    # Must match the name used in extract_and_cache
    'neg_variant': 'Baseline_CLIP',    # Can be DEO_CLIP if you cached that,,
    'pos_config': base_config,
    'neg_config': base_config,
    'split_ratio': 0.8,                # Train/Test split for the classifier
    'val_split': 0.2,                  # Internal validation for the classifier
    'alpha': 0.5,                      # Steering strength (from the paper)
    'lr': 0.01,                        # Classifier learning rate
    'epochs': 500,                      # Classifier training epochs
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}
run_paper_negation_experiment(exp_config)


--- Found cached features: ./embeddings_cache/Baseline_CLIP_43e228572a9931f231278079a4d4f81b.pt ---
--- Found cached features: ./embeddings_cache/Baseline_CLIP_43e228572a9931f231278079a4d4f81b.pt ---
Epoch 001 | Train Acc: 0.8357 | Loss: 0.49972 | Val Acc: 0.8857 | Loss: 0.38424 | LR: 1.0e-02
Epoch 050 | Train Acc: 0.9467 | Loss: 0.14977 | Val Acc: 0.9424 | Loss: 0.15513 | LR: 1.0e-02
Epoch 100 | Train Acc: 0.9479 | Loss: 0.14704 | Val Acc: 0.9460 | Loss: 0.15349 | LR: 5.0e-03
Epoch 150 | Train Acc: 0.9490 | Loss: 0.14614 | Val Acc: 0.9460 | Loss: 0.15338 | LR: 3.1e-04
Epoch 200 | Train Acc: 0.9493 | Loss: 0.14610 | Val Acc: 0.9469 | Loss: 0.15339 | LR: 2.0e-05
Epoch 250 | Train Acc: 0.9493 | Loss: 0.14610 | Val Acc: 0.9469 | Loss: 0.15339 | LR: 6.1e-07
Epoch 300 | Train Acc: 0.9493 | Loss: 0.14610 | Val Acc: 0.9469 | Loss: 0.15339 | LR: 1.9e-08
Epoch 350 | Train Acc: 0.9493 | Loss: 0.14610 | Val Acc: 0.9469 | Loss: 0.15339 | LR: 1.9e-08
Epoch 400 | Train Acc: 0.9493 | Loss: 0.14610 | 

{'W_dir': tensor([[-0.0337,  0.1397,  0.0899, -0.0423, -0.0114,  0.0610,  0.0434,  0.0326,
          -0.0064,  0.0928,  0.0440, -0.0340, -0.0233,  0.0319,  0.0504,  0.0081,
          -0.0286, -0.0339,  0.0129,  0.0689,  0.0367,  0.0061, -0.0338, -0.0492,
          -0.0934,  0.0330,  0.0268, -0.0156,  0.0403, -0.0396, -0.0041,  0.0332,
           0.0426,  0.0286, -0.0579,  0.0027,  0.0009,  0.0162,  0.0290, -0.0045,
           0.0361, -0.0411, -0.0510, -0.1013, -0.0867, -0.0468, -0.0380,  0.0374,
           0.0421, -0.0904, -0.0062,  0.0572, -0.0180,  0.0828,  0.0634, -0.0977,
           0.0223,  0.0555, -0.0825,  0.0479,  0.0346, -0.0346,  0.0778, -0.0086,
           0.0585, -0.0447,  0.0156,  0.0413,  0.0654,  0.0659, -0.0096, -0.0476,
           0.0115,  0.0552,  0.1126,  0.0529,  0.0676, -0.0201,  0.0535, -0.0712,
          -0.0002,  0.0260,  0.0493, -0.0458, -0.0045, -0.0122, -0.0437, -0.0406,
          -0.0689,  0.0598, -0.0595, -0.0619,  0.0168, -0.1446, -0.0002, -0.0147,
       

In [ ]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": 4}

# If you want to compare against your DEO features, use that config
# Otherwise, you can compare Baseline Pos vs Baseline Neg
exp_config = {
    'dataset': "NegationCLIP",
    'pos_variant': 'Baseline_CLIP_14000',    # Must match the name used in extract_and_cache
    'neg_variant': 'Baseline_CLIP_14000',    # Can be DEO_CLIP if you cached that,,
    'pos_config': base_config,
    'neg_config': base_config,
    'split_ratio': 0.8,                # Train/Test split for the classifier
    'val_split': 0.2,                  # Internal validation for the classifier
    'alpha': 0.5,                      # Steering strength (from the paper)
    'lr': 0.001,                        # Classifier learning rate
    'epochs': 250,                      # Classifier training epochs
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}
run_paper_negation_experiment(exp_config)


--- Found cached features: ./embeddings_cache/Baseline_CLIP_14000_120b7477eabe192a19c0baf063464394.pt ---
--- Found cached features: ./embeddings_cache/Baseline_CLIP_14000_120b7477eabe192a19c0baf063464394.pt ---


/tmp/ipykernel_273155/3879348306.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path)


Epoch 001 | Train Acc: 0.7823 | Loss: 0.63458 | Val Acc: 0.8158 | Loss: 0.58295 | LR: 1.0e-03
Epoch 010 | Train Acc: 0.9125 | Loss: 0.29828 | Val Acc: 0.9147 | Loss: 0.29040 | LR: 1.0e-03
Epoch 020 | Train Acc: 0.9269 | Loss: 0.22635 | Val Acc: 0.9292 | Loss: 0.22378 | LR: 1.0e-03
Epoch 030 | Train Acc: 0.9333 | Loss: 0.20018 | Val Acc: 0.9346 | Loss: 0.19986 | LR: 1.0e-03
Epoch 040 | Train Acc: 0.9363 | Loss: 0.18701 | Val Acc: 0.9371 | Loss: 0.18748 | LR: 1.0e-03
Epoch 050 | Train Acc: 0.9382 | Loss: 0.17907 | Val Acc: 0.9395 | Loss: 0.18025 | LR: 1.0e-03
Epoch 060 | Train Acc: 0.9403 | Loss: 0.17374 | Val Acc: 0.9404 | Loss: 0.17570 | LR: 1.0e-03
Epoch 070 | Train Acc: 0.9414 | Loss: 0.16986 | Val Acc: 0.9413 | Loss: 0.17216 | LR: 1.0e-03
Epoch 080 | Train Acc: 0.9421 | Loss: 0.16693 | Val Acc: 0.9417 | Loss: 0.16905 | LR: 1.0e-03
Epoch 090 | Train Acc: 0.9431 | Loss: 0.16471 | Val Acc: 0.9422 | Loss: 0.16734 | LR: 1.0e-03
Epoch 100 | Train Acc: 0.9437 | Loss: 0.16288 | Val Acc: 0.9

{'W_dir': tensor([[-4.9875e-02,  1.4757e-01,  9.8352e-02, -3.2421e-02, -6.8117e-03,
           4.2873e-02,  5.3455e-02,  2.4586e-02,  8.3640e-03,  8.2568e-02,
           3.3753e-02, -5.3776e-02, -2.7860e-02,  2.2298e-02,  4.3314e-02,
          -1.0039e-02, -2.2201e-02, -5.1941e-02,  1.6941e-02,  6.8541e-02,
           2.8037e-02,  1.7021e-02, -2.8479e-02, -5.2851e-02, -8.4929e-02,
           4.4060e-02,  1.7847e-02, -1.4459e-02,  3.2910e-02, -3.2115e-02,
          -1.3510e-02,  2.0968e-02,  3.6567e-02,  1.8782e-02, -6.2217e-02,
           1.2596e-02, -1.2342e-02,  1.4415e-02,  4.2199e-02, -8.8644e-03,
           1.2874e-02, -4.8770e-02, -5.3390e-02, -9.7077e-02, -7.2392e-02,
          -5.3760e-02, -2.2467e-02,  4.4372e-02,  4.4909e-02, -9.8772e-02,
           1.1487e-02,  3.9586e-02, -3.1782e-02,  8.8774e-02,  4.8807e-02,
          -7.6953e-02,  2.9152e-02,  6.0890e-02, -8.1577e-02,  5.8515e-02,
           3.3307e-02, -1.8102e-02,  9.0390e-02, -1.9043e-02,  6.5652e-02,
          -3.906

In [130]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": 4}

# If you want to compare against your DEO features, use that config
# Otherwise, you can compare Baseline Pos vs Baseline Neg
exp_config = {
    'dataset': "GEMINI_NEGATION",          # Must match the name used in extract_and_cache
    'pos_variant': 'Baseline_CLIP_Gemini',    # Must match the name used in extract_and_cache
    'neg_variant': 'Baseline_CLIP_Gemini',    # Can be DEO_CLIP if you cached that,,
    'pos_config': base_config,
    'neg_config': base_config,
    'split_ratio': 0.8,                # Train/Test split for the classifier
    'val_split': 0.2,                  # Internal validation for the classifier
    'alpha': 0.5,                      # Steering strength (from the paper)
    'lr': 0.01,                        # Classifier learning rate
    'epochs': 90,                      # Classifier training epochs
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}
run_paper_negation_experiment(exp_config)


--- Found cached features: ./embeddings_cache/Baseline_CLIP_Gemini_69e9c1072331f9acad27b90f6f251cc6.pt ---
--- Found cached features: ./embeddings_cache/Baseline_CLIP_Gemini_69e9c1072331f9acad27b90f6f251cc6.pt ---


/tmp/ipykernel_273155/3879348306.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path)


Epoch 001 | Train Acc: 0.9563 | Loss: 0.37781 | Val Acc: 0.9981 | Loss: 0.20322 | LR: 1.0e-02
Epoch 010 | Train Acc: 1.0000 | Loss: 0.01768 | Val Acc: 1.0000 | Loss: 0.01701 | LR: 1.0e-02
Epoch 020 | Train Acc: 1.0000 | Loss: 0.01239 | Val Acc: 1.0000 | Loss: 0.01232 | LR: 1.0e-02
Epoch 030 | Train Acc: 1.0000 | Loss: 0.01141 | Val Acc: 1.0000 | Loss: 0.01135 | LR: 1.0e-02
Epoch 040 | Train Acc: 1.0000 | Loss: 0.01105 | Val Acc: 1.0000 | Loss: 0.01095 | LR: 1.0e-02
Epoch 050 | Train Acc: 1.0000 | Loss: 0.01081 | Val Acc: 1.0000 | Loss: 0.01088 | LR: 1.0e-02
Epoch 060 | Train Acc: 1.0000 | Loss: 0.01075 | Val Acc: 1.0000 | Loss: 0.01064 | LR: 1.0e-02
Epoch 070 | Train Acc: 1.0000 | Loss: 0.01064 | Val Acc: 1.0000 | Loss: 0.01062 | LR: 1.0e-02
Epoch 080 | Train Acc: 1.0000 | Loss: 0.01061 | Val Acc: 1.0000 | Loss: 0.01056 | LR: 1.0e-02
Epoch 090 | Train Acc: 1.0000 | Loss: 0.01058 | Val Acc: 1.0000 | Loss: 0.01047 | LR: 1.0e-02

NEGATION EXPERIMENT: Baseline_CLIP_Gemini
Classifier Traini

{'W_dir': tensor([[ 0.0478,  0.0692,  0.0325, -0.0421, -0.0004,  0.0423,  0.0504,  0.0385,
           0.0332,  0.0775,  0.0695, -0.0452,  0.0738, -0.0570,  0.0496,  0.0535,
          -0.0618, -0.0624, -0.0474, -0.0453,  0.0304,  0.0224, -0.0596, -0.0292,
          -0.0629,  0.0322,  0.0088,  0.0185, -0.0432,  0.0253, -0.0374,  0.0503,
           0.0379, -0.0062, -0.0588, -0.0580, -0.0170,  0.0088,  0.0649, -0.0005,
          -0.0491,  0.0074, -0.0531, -0.0306, -0.0190, -0.0577,  0.0354,  0.0369,
          -0.0367, -0.0663,  0.0524, -0.0478, -0.0102,  0.0347,  0.0404, -0.0162,
           0.0325, -0.0302, -0.0627,  0.0592,  0.0033,  0.0399,  0.0476, -0.0706,
           0.0365, -0.0592,  0.0463, -0.0263,  0.0441,  0.0676,  0.0764, -0.0674,
           0.0608,  0.0530,  0.0479,  0.0579, -0.0170,  0.0689, -0.0168,  0.0578,
          -0.0539,  0.0125, -0.0143, -0.0483,  0.0354,  0.0484,  0.0142,  0.0586,
          -0.0189,  0.0257, -0.0586, -0.0375,  0.0454, -0.0637,  0.0057,  0.0196,
       

In [22]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": 4}

# If you want to compare against your DEO features, use that config
# Otherwise, you can compare Baseline Pos vs Baseline Neg
exp_config = {
    'dataset': "COCOValLlama",
    'pos_variant': 'Baseline_CLIP_COCOValLlama',    # Must match the name used in extract_and_cache
    'neg_variant': 'Baseline_CLIP_COCOValLlama',    # Can be DEO_CLIP if you cached that,,
    'pos_config': base_config,
    'neg_config': base_config,
    'split_ratio': 0.8,                # Train/Test split for the classifier
    'val_split': 0.2,                  # Internal validation for the classifier
    'alpha': 0.5,                      # Steering strength (from the paper)
    'lr': 0.001,                        # Classifier learning rate
    'epochs': 250,                      # Classifier training epochs
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}
run_paper_negation_experiment(exp_config)


--- Found cached features: ./embeddings_cache/Baseline_CLIP_COCOValLlama_962e837498b19128693aeeae96e7968a.pt ---
--- Found cached features: ./embeddings_cache/Baseline_CLIP_COCOValLlama_962e837498b19128693aeeae96e7968a.pt ---


/tmp/ipykernel_368465/3879348306.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path)


Epoch 001 | Train Acc: 0.8000 | Loss: 0.67550 | Val Acc: 0.8224 | Loss: 0.65778 | LR: 1.0e-03
Epoch 010 | Train Acc: 0.8185 | Loss: 0.50592 | Val Acc: 0.8192 | Loss: 0.49357 | LR: 1.0e-03
Epoch 020 | Train Acc: 0.8197 | Loss: 0.44710 | Val Acc: 0.8171 | Loss: 0.43517 | LR: 1.0e-03
Epoch 030 | Train Acc: 0.8199 | Loss: 0.42294 | Val Acc: 0.8219 | Loss: 0.41151 | LR: 1.0e-03
Epoch 040 | Train Acc: 0.8214 | Loss: 0.41007 | Val Acc: 0.8214 | Loss: 0.39928 | LR: 1.0e-03
Epoch 050 | Train Acc: 0.8213 | Loss: 0.40199 | Val Acc: 0.8203 | Loss: 0.39178 | LR: 1.0e-03
Epoch 060 | Train Acc: 0.8213 | Loss: 0.39591 | Val Acc: 0.8208 | Loss: 0.38643 | LR: 1.0e-03
Epoch 070 | Train Acc: 0.8219 | Loss: 0.39223 | Val Acc: 0.8235 | Loss: 0.38217 | LR: 1.0e-03
Epoch 080 | Train Acc: 0.8230 | Loss: 0.38839 | Val Acc: 0.8219 | Loss: 0.37940 | LR: 1.0e-03
Epoch 090 | Train Acc: 0.8235 | Loss: 0.38445 | Val Acc: 0.8235 | Loss: 0.37604 | LR: 1.0e-03
Epoch 100 | Train Acc: 0.8248 | Loss: 0.38212 | Val Acc: 0.8

{'W_dir': tensor([[ 9.5403e-04,  6.7881e-02, -3.9020e-02, -1.7455e-02,  3.3606e-03,
          -2.1995e-02, -6.1301e-03, -2.4265e-03,  2.9925e-03,  2.6213e-02,
           2.1023e-02,  5.4955e-02, -1.6331e-03,  4.0915e-03,  3.4743e-03,
           7.9042e-02, -2.6923e-02, -3.7167e-02, -1.0291e-01, -5.8891e-02,
           1.5728e-02, -1.0144e-02,  2.3953e-02, -5.4515e-02, -4.6727e-02,
          -4.0167e-04,  5.8718e-02,  7.6490e-02, -3.9569e-02,  4.2713e-02,
           3.1980e-02, -6.1596e-03, -7.3737e-03,  2.0750e-02, -8.3922e-02,
          -2.4090e-02,  1.4483e-02,  3.1238e-04,  7.0253e-02, -2.1948e-02,
           3.3813e-02, -4.7394e-02, -4.6967e-02,  1.3066e-02,  9.9216e-03,
          -4.8185e-02, -1.4981e-02,  9.4263e-02, -2.0897e-02, -8.1112e-02,
           1.6467e-02,  2.8654e-02,  1.6194e-02, -7.9190e-03,  6.6105e-02,
          -2.3923e-02,  8.3016e-02,  4.1751e-03, -6.1464e-02,  2.6076e-02,
          -4.3256e-03,  3.9741e-02,  5.6283e-02,  1.1965e-02,  1.1283e-02,
          -1.909

In [ ]:
# ============================================================================
# MODEL COMPARISON - NegRefCOCOg Dataset
# ============================================================================

# Load NegRefCOCOg dataset and model components
dataset = NegRefCOCOgDataset("NegRefCOCOg.json", max_samples=7000)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

# Load baseline CLIP model
base_model, preprocess = clip.load("ViT-B/32", device=device)
base_model.eval()

# Load learned negation direction
negation_vector_path = "learned_vectors/NegationCLIP_Baseline_CLIP_Baseline_CLIP_cb77e8a9_negation_vector.pt"
w_dir = load_negation_direction(negation_vector_path)
steered_model = NegationSteeredCLIP(base_model, w_dir=w_dir, alpha=0.5).to(device)
steered_model.eval()

# Evaluate using modular adapter
adapter = PairwiseModelAdapter("Steered CLIP", steered_model, device=device, text_mode="tokenized")
result = evaluate_pairwise_preference(adapter, loader, preprocess, device)

print(f"✓ Model Comparison Result: {result['model']} score={result['avg_score']:.4f}")

/tmp/ipykernel_273155/3142618687.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(vector_path, map_location="cpu")
/tmp/ipykernel_273155/3142618687.p


=== NegCOCOReg Pairwise Preference Results ===
DEO + Original CLIP      score=0.6068 (267/440)
DEO + NegationCLIP       score=0.6205 (273/440)


In [ ]:
# ============================================================================
# MODEL COMPARISON - Alternative Config
# ============================================================================

# Re-evaluate with different DEO weights
adapter2 = PairwiseModelAdapter("Steered CLIP (Alt)", steered_model, device=device, text_mode="tokenized")
result2 = evaluate_pairwise_preference(adapter2, loader, preprocess, device)

print(f"✓ Alternative config result: {result2['model']} score={result2['avg_score']:.4f}")

In [ ]:
# ============================================================================
# MODEL COMPARISON - Gemini Negation Dataset
# ============================================================================

# Load with Gemini negation vector
negation_vector_path_gemini = "learned_vectors/GEMINI_NEGATION_Baseline_CLIP_Gemini_Baseline_CLIP_Gemini_d100a7b8_negation_vector.pt"
w_dir_gemini = load_negation_direction(negation_vector_path_gemini)
steered_model_gemini = NegationSteeredCLIP(base_model, w_dir=w_dir_gemini, alpha=0.5).to(device)
steered_model_gemini.eval()

# Evaluate
adapter_gemini = PairwiseModelAdapter("Steered CLIP (Gemini)", steered_model_gemini, device=device, text_mode="tokenized")
result_gemini = evaluate_pairwise_preference(adapter_gemini, loader, preprocess, device)

print(f"✓ Gemini result: {result_gemini['model']} score={result_gemini['avg_score']:.4f}")

/tmp/ipykernel_273155/3810465031.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(vector_path, map_location="cpu")
/tmp/ipykernel_273155/3810465031.p


=== NegCOCOReg Pairwise Preference Results ===
Steered CLIP             score=0.5523 (243/440)


In [20]:
base_config = {"type": "vanilla", "arch": "ViT-B/32", "layer": 4}

# Negation experiment for COCOValLlama dataset
exp_config_cocovallama = {
    'dataset': "COCOValLlama",
    'pos_variant': 'Baseline_CLIP_COCOValLlama',
    'neg_variant': 'Baseline_CLIP_COCOValLlama',
    'pos_config': base_config,
    'neg_config': base_config,
    'split_ratio': 0.8,
    'val_split': 0.2,
    'alpha': 0.5,
    'lr': 0.001,
    'epochs': 250,
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}

print("Running negation experiment on COCOValLlama dataset...")
coco_negation_output = run_paper_negation_experiment(exp_config_cocovallama)


Running negation experiment on COCOValLlama dataset...


TypeError: 'NoneType' object is not subscriptable

In [ ]:
def evaluate_negation_steering_on_text(z_pos, z_neg, W_dir, alpha_values=[0.0, 0.3, 0.5, 0.7, 1.0], device="cuda"):
    """
    Evaluate how negation steering affects text-text similarity.
    For each alpha, compute how much negative texts are pushed away from positive texts.
    
    Metrics:
    - Original sim (pos vs neg, no steering)
    - Steered sim (pos vs neg, with steering)
    - Improvement (how much the gap increases)
    """
    z_pos = z_pos.float().to(device)
    z_neg = z_neg.float().to(device)
    W_dir = W_dir.float().to(device)
    
    results = {"alpha": [], "baseline_sim": [], "steered_sim": [], "improvement": []}
    
    # Baseline: no steering
    z_pos_norm = F.normalize(z_pos, p=2, dim=-1)
    z_neg_norm = F.normalize(z_neg, p=2, dim=-1)
    baseline_sim = torch.cosine_similarity(z_pos_norm, z_neg_norm).mean().item()
    
    print("\n=== Text Steering Evaluation on COCOValLlama ===")
    print(f"Baseline (no steering)  | Sim(pos,neg) = {baseline_sim:.4f}\n")
    
    for alpha in alpha_values:
        z_pos_steered = steer_embeddings(z_pos, W_dir, alpha)
        z_pos_steered_norm = F.normalize(z_pos_steered, p=2, dim=-1)
        steered_sim = torch.cosine_similarity(z_pos_steered_norm, z_neg_norm).mean().item()
        improvement = baseline_sim - steered_sim  # Higher = better (lower similarity)
        
        results["alpha"].append(alpha)
        results["baseline_sim"].append(baseline_sim)
        results["steered_sim"].append(steered_sim)
        results["improvement"].append(improvement)
        
        print(f"Alpha = {alpha:.2f}  | Sim(pos,neg) = {steered_sim:.4f} | Gap Improvement = {improvement:+.4f}")
    
    print("="*50)
    return results


In [ ]:
# Evaluate steering on COCOValLlama test set
cache_manager = FeatureCache()
pos_path_coco = cache_manager.get_cache_path("Baseline_CLIP_COCOValLlama", "COCOValLlama", base_config)
pos_data_coco = cache_manager.load(pos_path_coco)

if pos_data_coco is not None:
    z_pos_test = pos_data_coco['pos_text'][int(0.8 * len(pos_data_coco['pos_text'])):].to(device)
    z_neg_test = pos_data_coco['neg_text'][int(0.8 * len(pos_data_coco['neg_text'])):].to(device)
    
    # Use the learned W_dir from coco_negation_output
    W_dir_coco = coco_negation_output['W_dir'].to(device)
    
    steering_results = evaluate_negation_steering_on_text(
        z_pos=z_pos_test,
        z_neg=z_neg_test,
        W_dir=W_dir_coco,
        alpha_values=[0.0, 0.3, 0.5, 0.7, 1.0],
        device=device
    )
else:
    print("COCOValLlama features not found. Please run feature extraction first.")


In [ ]:
# Optional: Compare different layers on COCOValLlama
print("Extracting features at final layer (-1) for COCOValLlama comparison...")
base_config_final = {"type": "vanilla", "arch": "ViT-B/32", "layer": -1}
base_model_final, _ = clip.load("ViT-B/32", device=device)

# Reload the COCOValLlama dataset
dataset_coco = COCOValLlamaDataset("COCO_val_mcq_llama3.1_rephrased.json", max_samples=14000)
loader_coco = DataLoader(dataset_coco, batch_size=128, shuffle=False)

base_features_final = extract_and_cache(
    base_model_final, 
    loader_coco, 
    clip.tokenize, 
    "Baseline_CLIP_COCOValLlama_Final", 
    base_config_final, 
    dataset="COCOValLlama",
    device=device
)

print("Feature extraction at final layer complete.")


In [ ]:
# Run negation experiment at final layer for COCOValLlama
base_config_final = {"type": "vanilla", "arch": "ViT-B/32", "layer": -1}

exp_config_cocovallama_final = {
    'dataset': "COCOValLlama",
    'pos_variant': 'Baseline_CLIP_COCOValLlama_Final',
    'neg_variant': 'Baseline_CLIP_COCOValLlama_Final',
    'pos_config': base_config_final,
    'neg_config': base_config_final,
    'split_ratio': 0.8,
    'val_split': 0.2,
    'alpha': 0.5,
    'lr': 0.001,
    'epochs': 250,
    'batch_size': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}

print("Running negation experiment on COCOValLlama at final layer...")
coco_negation_output_final = run_paper_negation_experiment(exp_config_cocovallama_final)


In [ ]:
# Summary comparison of layer performance on COCOValLlama
print("\n" + "="*60)
print("COCOVALLAMA NEGATION RESULTS SUMMARY")
print("="*60)

print("\n[Layer 4 (Mid-layer)]")
print(f"  Test Accuracy:     {coco_negation_output['test_acc']:.2%}")
print(f"  Baseline Sim:      {coco_negation_output['baseline_sim']:.4f}")
print(f"  Steered Sim:       {coco_negation_output['result_sim']:.4f}")
print(f"  Total Gain:        {coco_negation_output['gain']:+.4f}")

print("\n[Layer -1 (Final)]")
print(f"  Test Accuracy:     {coco_negation_output_final['test_acc']:.2%}")
print(f"  Baseline Sim:      {coco_negation_output_final['baseline_sim']:.4f}")
print(f"  Steered Sim:       {coco_negation_output_final['result_sim']:.4f}")
print(f"  Total Gain:        {coco_negation_output_final['gain']:+.4f}")

print("\n" + "="*60)
print("Analysis:")
print("-"*60)
if coco_negation_output['test_acc'] > coco_negation_output_final['test_acc']:
    diff = (coco_negation_output['test_acc'] - coco_negation_output_final['test_acc']) * 100
    print(f"✓ Layer 4 achieves {diff:.1f}% higher test accuracy")
else:
    diff = (coco_negation_output_final['test_acc'] - coco_negation_output['test_acc']) * 100
    print(f"✓ Final layer achieves {diff:.1f}% higher test accuracy")

gain_diff = coco_negation_output['gain'] - coco_negation_output_final['gain']
print(f"✓ Layer 4 gain diff: {gain_diff:+.4f} vs Final layer")
print("="*60)


## Step 1: Extract LLM Decomposition from Negated Retrieval Dataset

Extract and cache LLM decompositions for the CSV negation dataset captions.

In [25]:
# ============================================================================
# Load Negated Retrieval CSV Dataset and Extract LLM Decompositions
# ============================================================================

import pandas as pd
import json
from pathlib import Path
import ast

# Load the CSV dataset
csv_path = "COCO_val_negated_retrieval_llama3.1_rephrased_affneg_true.csv"
print(f"Loading dataset from {csv_path}...")
df = pd.read_csv(csv_path)
print(f"✓ Loaded {len(df)} samples")

# Display first few rows
print("\nDataset preview:")
print(df.head(2))

# ============================================================================
# Initialize SubQuery Extractor using predefined class
# ============================================================================

print("\n" + "="*70)
print("EXTRACTING LLM DECOMPOSITIONS FOR NEGATION CAPTIONS")
print("="*70)

# Initialize LLM client if not already available
if 'qwen_client' not in locals():
    print("\nInitializing Qwen LLM client...")
    qwen_client = LocalQwenClient(model_id="Qwen/Qwen2.5-1.5B-Instruct", device=device)
    print("✓ LLM client initialized")

# Initialize SubQuery Extractor using the predefined class
print("\nInitializing SubQuery Extractor...")
extractor = SubQueryExtractor(
    llm_client=qwen_client,
    model_name=qwen_client.model_id,
    prompt_version="v3",
    dataset_name="NegatedRetrieval"
)
print(f"✓ SubQuery Extractor initialized (cache: {extractor.cache_dir})")

# ============================================================================
# Collect all captions from CSV and extract decompositions
# ============================================================================

print("\nCollecting all captions from dataset...")
all_captions = []
caption_metadata = {}  # Map caption to image_id for later retrieval

for idx, row in df.iterrows():
    image_id = row['image_id']
    captions = ast.literal_eval(row['captions']) if isinstance(row['captions'], str) else row['captions']
    
    for cap_idx, caption in enumerate(captions):
        all_captions.append(caption)
        caption_metadata[caption] = {
            'image_id': image_id,
            'caption_idx': cap_idx,
            'positive_objects': ast.literal_eval(row['positive_objects']) if isinstance(row['positive_objects'], str) else row['positive_objects'],
            'negative_objects': ast.literal_eval(row['negative_objects']) if isinstance(row['negative_objects'], str) else row['negative_objects']
        }

print(f"✓ Collected {len(all_captions)} captions from {len(df)} images")

# ============================================================================
# Extract decompositions using SubQueryExtractor in batch mode
# ============================================================================

print(f"\nExtracting decompositions using SubQueryExtractor (batch size: 16)...")
batch_size = 16
all_decompositions = []

# Process captions in batches
for i in tqdm(range(0, len(all_captions), batch_size), desc="Processing Batches"):
    batch_captions = all_captions[i : i + batch_size]
    
    # Use the predefined class's batch method
    batch_results = extractor.get_decomposition_batch(batch_captions)
    all_decompositions.extend(batch_results)

print(f"\n✓ Extracted {len(all_decompositions)} decompositions")

# ============================================================================
# Build cached result structure organized by image_id
# ============================================================================

print("\nOrganizing decompositions by image_id...")
decompositions_cache = {}

for caption, decomposition in zip(all_captions, all_decompositions):
    metadata = caption_metadata[caption]
    image_id = metadata['image_id']
    cap_idx = metadata['caption_idx']
    
    # Create image entry if not exists
    if image_id not in decompositions_cache:
        decompositions_cache[image_id] = {
            'positive_objects': metadata['positive_objects'],
            'negative_objects': metadata['negative_objects'],
            'caption_decompositions': {}
        }
    
    # Store decomposition
    decompositions_cache[image_id]['caption_decompositions'][cap_idx] = decomposition

print(f"✓ Organized {len(decompositions_cache)} images")

# ============================================================================
# Display sample decompositions and statistics
# ============================================================================

print("\n" + "="*70)
print("SAMPLE DECOMPOSITIONS")
print("="*70)

sample_image_id = df.iloc[0]['image_id']
sample_decomp = decompositions_cache[sample_image_id]

print(f"\nImage ID: {sample_image_id}")
print(f"Positive objects: {sample_decomp['positive_objects']}")
print(f"Negative objects: {sample_decomp['negative_objects']}")
print(f"\nCaption Decompositions:")
for cap_idx, decomp in sample_decomp['caption_decompositions'].items():
    print(f"\n  Caption {cap_idx}:")
    if 'positives' in decomp and 'negatives' in decomp:
        print(f"    Positives: {decomp['positives']}")
        print(f"    Negatives: {decomp['negatives']}")
    else:
        print(f"    Result: {json.dumps(decomp, indent=6)}")

print("\n" + "="*70)
print("EXTRACTION SUMMARY")
print("="*70)
print(f"✓ Total images processed: {len(decompositions_cache)}")
print(f"✓ Total captions processed: {len(all_captions)}")
print(f"✓ Cache location: {extractor.cache_dir}")
print(f"✓ LLM Decomposition Extraction Complete!")
print(f"✓ Ready for Experiment 2: MLLM-as-Judge Evaluation")

Loading dataset from COCO_val_negated_retrieval_llama3.1_rephrased_affneg_true.csv...
✓ Loaded 5000 samples

Dataset preview:
                                    positive_objects  \
0  ['person', 'bottle', 'cup', 'knife', 'spoon', ...   
1  ['banana', 'orange', 'chair', 'potted plant', ...   

              negative_objects                                   filepath  \
0     ['chair', 'fork', 'car']  data/coco/images/val2017/000000397133.jpg   
1  ['person', 'cup', 'bottle']  data/coco/images/val2017/000000037777.jpg   

   image_id                                           captions  
0    397133  ['A man in a kitchen is making pizzas, but the...  
1     37777  ['No cup can be seen; however, the dining tabl...  

EXTRACTING LLM DECOMPOSITIONS FOR NEGATION CAPTIONS

Initializing SubQuery Extractor...
✓ SubQuery Extractor initialized (cache: ./llm_cache/NegatedRetrieval/v3)

✓ Collected 25014 captions from 5000 images

Extracting decompositions using SubQueryExtractor (batch size: 16)...

Processing Batches:   1%|          | 11/1564 [00:43<2:43:47,  6.33s/it]

Extraction Error for: No handbag is present in the i... trying simple fallback.


Processing Batches:   1%|          | 18/1564 [01:15<2:38:55,  6.17s/it]

Extraction Error for: No bird is visible in the imag... trying simple fallback.


...

Extraction Error for: No handbag is present; instead... trying simple fallback.


Processing Batches: 100%|█████████▉| 1562/1564 [1:30:48<00:14,  7.45s/it]

Extraction Error for: No TV is visible in the image,... trying simple fallback.


Processing Batches: 100%|██████████| 1564/1564 [1:30:52<00:00,  3.49s/it]


✓ Extracted 25014 decompositions

Organizing decompositions by image_id...
✓ Organized 5000 images

SAMPLE DECOMPOSITIONS

Image ID: 397133
Positive objects: ['person', 'bottle', 'cup', 'knife', 'spoon', 'bowl', 'broccoli', 'carrot', 'dining table', 'oven', 'sink']
Negative objects: ['chair', 'fork', 'car']

Caption Decompositions:

  Caption 0:
    Positives: ['man in kitchen', 'making pizzas', 'kitchen setting', 'pizza-making process', 'food preparation', 'cooking activity']
    Negatives: ['chair', 'sight of chair', 'lack of chair']

  Caption 1:
    Positives: ['a man in an apron', 'stands in front of an oven', 'ovens', 'pans', 'bakeware', 'nearby']
    Negatives: ['chair']

  Caption 2:
    Positives: ['baker working in the kitchen', 'rolling out dough', 'kitchen activities']
    Negatives: ['forks present', 'no forks']

  Caption 3:
    Positives: ['person standing by a stove in the kitchen', 'stove in the kitchen', 'kitchen setting', 'stand near the stove', 'kitchen activity']


In [ ]:
import json
import shutil
from pathlib import Path
from tqdm import tqdm

# Configuration
json_path = Path("existence.json")
output_dir = Path("images")
source_dirs = [
    Path("visual7w_images"),  # From unzipped visual7w_images.zip
    Path("v7w"),
    Path("."),
]

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Output directory: {output_dir.absolute()}")

# Load and parse existence.json
with open(json_path, 'r') as f:
    data = json.load(f)

# Extract unique image filenames
image_files = sorted(list(set(
    item['image_file'] for item in data.values() if 'image_file' in item
)))
print(f"✓ Found {len(image_files)} unique images to copy")

# Copy images
copied = 0
missing = 0

for image_file in tqdm(image_files, desc="Copying"):
    for source_dir in source_dirs:
        source_path = source_dir / image_file
        if source_path.exists():
            shutil.copy2(source_path, output_dir / image_file)
            copied += 1
            break
    else:
        missing += 1
        print(f"⚠ Missing: {image_file}")

print(f"\n✓ Copied: {copied}/{len(image_files)} images")
if missing > 0:
    print(f"⚠ Missing: {missing} images")